# FB15k-237 entity typing MVP

### 1. Load the data
Training triples and Level-1 entity types (80/20 split) from [preprocessing](fb15k_preprocessing.ipynb).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from pykeen.models import TransE
from pykeen.training import SLCWATrainingLoop
from pykeen.triples import TriplesFactory
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

data_dir = Path("data/processed/fb15k-237")
if Path("fb15k-237_train.tsv").is_file():
    data_dir = Path("processed/fb15k-237")
if not all((data_dir / name).is_file() for name in ("train.tsv", "entity_split.tsv")):
    raise FileNotFoundError("Run fb15k_preprocessing.ipynb first.")

triples = TriplesFactory.from_path(data_dir / "train.tsv")
entities = pd.read_csv(data_dir / "entity_split.tsv", sep="\t")
level_one = entities.loc[entities["experiment"].eq("Level-1")].copy()
level_one["entity_id"] = level_one["entity"].map(triples.entity_to_id).astype(int)

### 2. Train TransE
Three epochs to learn a 32-dimensional vector per entity.

In [2]:
model = TransE(triples_factory=triples, embedding_dim=32, random_seed=37).cpu()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
training = SLCWATrainingLoop(model=model, triples_factory=triples, optimizer=optimizer)
training.train(
    triples_factory=triples, num_epochs=3, batch_size=1024,
    use_tqdm=False, use_tqdm_batch=False, pin_memory=False,
)

model.eval()
with torch.no_grad():
    embeddings = model.entity_representations[0]().numpy()

### 3. Predict entity types
Compare logistic regression with the most-frequent-type baseline using macro-F1.

In [3]:
train = level_one.loc[level_one["split"].eq("train")]
test = level_one.loc[level_one["split"].eq("test")]
x_train = embeddings[train["entity_id"].to_numpy()]
x_test = embeddings[test["entity_id"].to_numpy()]

label_encoder = MultiLabelBinarizer()
y_train = label_encoder.fit_transform(train["classes"].str.split(","))
y_test = label_encoder.transform(test["classes"].str.split(","))

classifier = OneVsRestClassifier(LogisticRegression())
predicted = classifier.fit(x_train, y_train).predict(x_test)
baseline = np.zeros_like(y_test)
baseline[:, y_train.sum(axis=0).argmax()] = 1

scores = pd.Series({
    "Most frequent type": f1_score(y_test, baseline, average="macro"),
    "TransE + logistic regression": f1_score(y_test, predicted, average="macro"),
}, name="Macro-F1").to_frame()
scores.round(3)

,Macro-F1
Most frequent type,0.231
TransE + logistic regression,0.919
